# CritterGym — GPU throughput bench (M4-EC3)

Measures how many **environment steps/second** CritterGym's vectorized JAX env runs on a
**free cloud NVIDIA GPU**. The project's CPU is already fast; this gets the GPU number for
the M4-EC3 target (**≥10M steps/s**), which the maintainer's Apple-Silicon Mac can't produce
(jax-metal crashes on the fused rollout). You only need a Google login.

**Steps (clicks only — no coding):**
1. Menu **Runtime → Change runtime type → Hardware accelerator: GPU** (a free **T4** is fine), Save.
2. Run every cell top-to-bottom (**Runtime → Run all**, or ▶ each cell).
3. When it finishes, **copy the whole output of the last two cells** and paste it back to Claude.

_Honest notes: a free T4 is modest but easily vectorizes; numbers vary per run/GPU. If the repo is
private, fill the token cell. Nothing here is verified on GPU by the author — it's the standard
Colab clone+pip+run pattern; the bench itself was sanity-checked on CPU._

## 1. Confirm a GPU is attached
If this errors or shows no GPU, redo step 1 (Runtime → GPU) and re-run.

In [ ]:
!nvidia-smi

## 2. (Only if the repo is PRIVATE) paste a GitHub token
Leave it blank and just run if the repo is public. The token is read with `getpass`
(not stored in the notebook) and only used to clone.

In [ ]:
from getpass import getpass

TOKEN = getpass("GitHub token (blank if public, then press Enter): ").strip()

## 3. Clone the repo

In [ ]:
import os
import subprocess

REPO = "https://github.com/anolysius/creature-rl-env.git"
url = REPO if not TOKEN else REPO.replace("https://", f"https://{TOKEN}@")
if not os.path.isdir("creature-rl-env"):
    subprocess.run(["git", "clone", "--depth", "1", url], check=True)
print("cloned:", os.path.isdir("creature-rl-env"))

## 4. Install JAX (CUDA GPU build) + the package
`jax[cuda12]` pulls the GPU wheels; the package install adds only numpy/gymnasium
(it does **not** touch the GPU jax).

In [ ]:
!pip install -q -U 'jax[cuda12]'
!pip install -q -e creature-rl-env

In [ ]:
import jax  # noqa: E402

print("JAX backend:", jax.default_backend())
print("devices:", jax.devices())

## 5. Run the GPU-fair throughput bench (fused `lax.scan` rollout)
Prints numpy / jax-single / jax-vmap steps/s for the overworld slice and the
full-episode env, over a batch sweep. **The vmap rows are the GPU number.**

In [ ]:
!cd creature-rl-env && python scripts/gpu_bench.py

## 6. Training throughput (env-steps/s of a real PPO rollout on GPU)
A short on-device PPO run — reports env-steps/s the RL loop actually achieves on the GPU.

In [ ]:
from critter_gym.jax_train import PPOConfig, train_ppo

cfg = PPOConfig(batch=2048, rollout_len=32, iters=30, hidden=64)
res = train_ppo(tuple(range(2048)), cfg, seed=0)
print(f"train env-steps/s: {res.env_steps_per_s:,.0f}  "
      f"(total {res.total_env_steps:,} steps in {res.wall_time_s:.2f}s)")

## 7. Done — copy the output of cells 5 and 6 and paste it back to Claude
Claude will record the GPU steps/s against the M4-EC3 target (≥10M steps/s) honestly,
whatever the number turns out to be.